# SafeSignal — GRPO Training Notebook

Training an LLM to learn optimal intervention timing
for child digital wellbeing using Group Relative Policy Optimisation.

**Key numbers to watch:**
- Random baseline: ~-45
- Silent baseline: ~+15.85
- Trained target: above +15.85

**Runtime:** T4 GPU (~25 min) or A100 (~8 min)

---

| Step | What happens |
|------|--------------|
| Cell 1 | Install dependencies |
| Cell 2 | Clone repo & check GPU |
| Cell 3 | Baseline evaluation (random + silent agents) |
| Cell 4 | GRPO training with live environment reward |
| Cell 5 | Generate result plots |

## Cell 1 — Install Dependencies

In [ ]:
# Upgrade pip quietly, then install everything needed
!pip install -q --upgrade pip
!pip install -q \
    'trl>=0.12.0' \
    transformers \
    datasets \
    peft \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    matplotlib \
    numpy

print('All dependencies installed.')

## Cell 2 — Clone Repo & Check GPU

In [ ]:
import os, sys

# Clone the repo (skip if already cloned)
if not os.path.exists('OpenENV-Environment'):
    !git clone https://github.com/Denisa1315/OpenENV-Environment.git

%cd OpenENV-Environment
sys.path.insert(0, '.')

# Confirm GPU
!nvidia-smi | head -12

## Cell 3 — Baseline Evaluation

Runs two reference policies for 200 episodes each:
- **Random agent** — picks actions uniformly at random  
- **Always-silent agent** — always chooses `OBSERVE_QUIETLY`

The trained model must beat the silent agent to show it learned something useful.

In [ ]:
import sys, os, json, random

sys.path.insert(0, '.')
from server.environment import SafeSignalEnvironment, ACTIONS

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

N_BASELINE = 200

# ── Random Agent ──────────────────────────────────────────────────────────
print(f'Running random agent ({N_BASELINE} episodes)...')
random_results = []
for ep in range(N_BASELINE):
    env    = SafeSignalEnvironment()
    state  = env.reset().observation
    done   = False
    ep_reward   = 0.0
    trust_traj  = []
    hidden_traj = []
    actions_taken = []
    while not done:
        action = random.choice(ACTIONS)
        step   = env.step(action)
        ep_reward += step.reward
        trust_traj.append(step.info['guardian_trust'])
        hidden_traj.append(step.info['hidden_state'])
        actions_taken.append(action)
        state = step.observation
        done  = step.done
    random_results.append({
        'episode': ep,
        'total_reward': round(ep_reward, 3),
        'final_hidden_state': step.info['hidden_state'],
        'final_guardian_trust': round(step.info['guardian_trust'], 3),
        'trust_trajectory': trust_traj,
        'hidden_trajectory': hidden_traj,
        'total_interventions': sum(1 for a in actions_taken if a != 'OBSERVE_QUIETLY'),
        'ended_safe': step.info['hidden_state'] == 'SAFE',
    })

random_avg      = sum(r['total_reward'] for r in random_results) / N_BASELINE
random_safe_pct = sum(1 for r in random_results if r['ended_safe']) / N_BASELINE * 100
print(f'  Random avg reward: {random_avg:+.2f}  ({random_safe_pct:.1f}% safe)')

baseline_data = {
    'policy': 'random',
    'n_episodes': N_BASELINE,
    'avg_reward': round(random_avg, 3),
    'pct_ended_safe': round(random_safe_pct, 1),
    'episodes': random_results,
}

# ── Always-Silent Agent ────────────────────────────────────────────────────
print(f'\nRunning silent agent ({N_BASELINE} episodes)...')
silent_results = []
for ep in range(N_BASELINE):
    env  = SafeSignalEnvironment()
    env.reset()
    done = False
    ep_reward   = 0.0
    trust_traj  = []
    hidden_traj = []
    while not done:
        step = env.step('OBSERVE_QUIETLY')
        ep_reward += step.reward
        trust_traj.append(step.info['guardian_trust'])
        hidden_traj.append(step.info['hidden_state'])
        done = step.done
    silent_results.append({
        'episode': ep,
        'total_reward': round(ep_reward, 3),
        'final_hidden_state': step.info['hidden_state'],
        'final_guardian_trust': round(step.info['guardian_trust'], 3),
        'trust_trajectory': trust_traj,
        'hidden_trajectory': hidden_traj,
        'total_interventions': 0,
        'ended_safe': step.info['hidden_state'] == 'SAFE',
    })

silent_avg      = sum(r['total_reward'] for r in silent_results) / N_BASELINE
silent_safe_pct = sum(1 for r in silent_results if r['ended_safe']) / N_BASELINE * 100
print(f'  Silent avg reward: {silent_avg:+.2f}  ({silent_safe_pct:.1f}% safe)')

silent_data = {
    'policy': 'silent',
    'n_episodes': N_BASELINE,
    'avg_reward': round(silent_avg, 3),
    'pct_ended_safe': round(silent_safe_pct, 1),
    'episodes': silent_results,
}

# ── Save ──────────────────────────────────────────────────────────────────
with open(f'{RESULTS_DIR}/baseline_rewards.json', 'w') as f:
    json.dump(baseline_data, f, indent=2)
with open(f'{RESULTS_DIR}/silent_rewards.json', 'w') as f:
    json.dump(silent_data, f, indent=2)

print(f'\nBaseline saved. Trained model must beat: {silent_avg:+.2f}')

## Cell 4 — GRPO Training

Trains TinyLlama-1.1B with **GRPO** (Group Relative Policy Optimisation) using HuggingFace TRL.

**What GRPO does here:**  
For each environment state prompt it generates `num_generations=4` candidate responses,  
scores them with the composable rubric reward function, then uses the relative advantage  
between responses to update the policy — no value network needed.

**Three-component reward:**
- Action correctness via composable rubrics (70%)
- Reasoning quality — did it mention trust, timing, archetype? (20%)
- Format compliance — action on final line (10%)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
import torch

MODEL_NAME       = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
MAX_SEQ_LEN      = 256   # shorter = faster generation
MAX_NEW_TOKENS   = 40    # generation is the bottleneck — keep low
N_EPISODES       = 20    # episodes to collect training prompts from
MAX_STEPS        = 200   # hard cap on training steps
CHECKPOINT_EVERY = 50
LOG_EVERY        = 2

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_4BIT = torch.cuda.is_available()
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

print(f'Device: {DEVICE}  |  4-bit: {USE_4BIT}  |  bf16: {USE_BF16}  |  fp16: {USE_FP16}')

In [ ]:
# ── Load Model ────────────────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

def load_model(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    kwargs = dict(low_cpu_mem_usage=True)
    if USE_4BIT:
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        kwargs['device_map'] = 'auto'
    else:
        kwargs['torch_dtype'] = torch.float32
    try:
        kwargs['attn_implementation'] = 'flash_attention_2'
        m = AutoModelForCausalLM.from_pretrained(name, **kwargs)
    except Exception:
        kwargs.pop('attn_implementation', None)
        m = AutoModelForCausalLM.from_pretrained(name, **kwargs)
    return m, tokenizer

print(f'Loading {MODEL_NAME} ...')
model, tokenizer = load_model(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05, bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Build Dataset from Live Environment ───────────────────────────────────
import random
from datasets import Dataset
from server.environment import SafeSignalEnvironment, ACTIONS
from training.prompt_builder import state_to_prompt

print(f'Generating prompts from {N_EPISODES} live environment episodes...')

training_samples = []
env = SafeSignalEnvironment()

for episode in range(N_EPISODES):
    result = env.reset()
    state  = result.observation
    done   = False
    while not done:
        training_samples.append({
            'prompt':         state_to_prompt(state),
            'hidden_state':   env.child.hidden_state,
            'state_snapshot': state.to_dict(),    # FIXED — was dict(state)
        })
        step  = env.step(random.choice(ACTIONS))
        state = step.observation
        done  = step.done

training_metadata = {
    i: {'hidden_state': s['hidden_state'], 'state_snapshot': s['state_snapshot']}
    for i, s in enumerate(training_samples)
}

dataset = Dataset.from_list([{'prompt': s['prompt']} for s in training_samples])
print(f'Dataset: {len(dataset)} prompts from {N_EPISODES} episodes')

In [ ]:
# ── Reward Function ───────────────────────────────────────────────────────
from training.prompt_builder import parse_action
from training.grpo_rewards import compute_grpo_reward

sample_index = [0]

def grpo_reward_fn(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        idx  = sample_index[0] % len(training_metadata)
        meta = training_metadata[idx]

        response_text = (
            completion[0]['content'] if isinstance(completion, list) else completion
        )

        state        = meta['state_snapshot']
        hidden_state = meta['hidden_state']

        temp_env = SafeSignalEnvironment(archetype=state.get('child_archetype', 'target'))
        temp_env.reset()
        temp_env.child.guardian_trust = state.get('guardian_trust', 0.8)
        temp_env.child.consecutive_ignored_alerts = state.get('consecutive_ignored_alerts', 0)
        guardian_response = temp_env.child.simulate_guardian_response(parse_action(response_text))

        reward = compute_grpo_reward(
            response_text=response_text, state=state,
            hidden_state=hidden_state, guardian_response=guardian_response,
        )
        rewards.append(float(reward))

    sample_index[0] += 1
    return rewards

print('Reward function ready.')

In [ ]:
# ── GRPO Config & Train ───────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

os.makedirs('./training/checkpoints', exist_ok=True)  # FIXED — ensure dir exists

grpo_config = GRPOConfig(
    num_generations             = 4,
    max_prompt_length           = MAX_SEQ_LEN,
    max_completion_length       = MAX_NEW_TOKENS,
    learning_rate               = 5e-6,
    per_device_train_batch_size = 2 if DEVICE == 'cuda' else 1,
    gradient_accumulation_steps = 4 if DEVICE == 'cuda' else 8,
    max_steps                   = MAX_STEPS,
    beta                        = 0.01,
    output_dir                  = './training/checkpoints',
    logging_steps               = LOG_EVERY,
    save_steps                  = CHECKPOINT_EVERY,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    bf16                        = USE_BF16,
    fp16                        = USE_FP16,
    dataloader_num_workers      = 2 if DEVICE == 'cuda' else 0,
    report_to                   = 'none',
)

trainer = GRPOTrainer(
    model            = model,
    reward_funcs     = grpo_reward_fn,
    args             = grpo_config,
    train_dataset    = dataset,
    processing_class = tokenizer,
)

print(f'Starting GRPO training...')
print(f'  Model:      {MODEL_NAME}')
print(f'  Device:     {DEVICE}')
print(f'  Prompts:    {len(dataset)}')
print(f'  Gens/step:  {grpo_config.num_generations}')
print(f'  Max steps:  {MAX_STEPS}')
print()

trainer.train()
trainer.save_model('./training/checkpoints/final')
print('Training complete.')

# ── Save training reward curve ─────────────────────────────────────────────
training_log = trainer.state.log_history
training_rewards = [
    entry.get('reward', None)
    for entry in training_log
    if 'reward' in entry
]

if training_rewards:
    with open('results/training_log.json', 'w') as f:
        json.dump({
            'log_history':            training_log,
            'reward_during_training': training_rewards,
        }, f, indent=2)
    print(f'Training log saved. Final training reward: {training_rewards[-1]:+.3f}')

In [ ]:
# ── Evaluate Trained Model ─────────────────────────────────────────────────
import json
from training.prompt_builder import state_to_prompt, parse_action

# FIXED — guard against running this cell before Cell 3
silent_avg = silent['avg_reward']

model.eval()
eval_results   = []
eval_trust     = []
eval_outcomes  = []
rubric_history = []

print('Evaluating trained model (100 episodes)...')

for episode in range(100):
    env    = SafeSignalEnvironment()
    state  = env.reset().observation
    done   = False
    ep_reward  = 0
    ep_rubrics = []

    while not done:
        inputs = tokenizer(
            state_to_prompt(state), return_tensors='pt',
            truncation=True, max_length=MAX_SEQ_LEN,
        ).to(DEVICE if not USE_4BIT else model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.1, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        step = env.step(parse_action(response))
        ep_reward += step.reward
        if step.rubric_scores:
            ep_rubrics.append(step.rubric_scores)
        state = step.observation
        done  = step.done

    eval_results.append(round(ep_reward, 3))
    eval_trust.append(round(step.info['guardian_trust'], 3))
    eval_outcomes.append(step.info['hidden_state'])

    if ep_rubrics:
        avg_rubric = {}
        for name in ['intervention_timing', 'guardian_trust', 'silence_intelligence']:
            scores = [r.get(name, {}).get('weighted_score', 0) for r in ep_rubrics if name in r]
            if scores:
                avg_rubric[name] = sum(scores) / len(scores)
        rubric_history.append(avg_rubric)

    if episode % 20 == 0:
        print(f'  Episode {episode:3d} | Avg reward: {sum(eval_results)/len(eval_results):+.2f}')

trained_avg      = sum(eval_results) / len(eval_results)
trained_safe_pct = sum(1 for o in eval_outcomes if o == 'SAFE') / 100 * 100

print(f'\n  Trained avg reward:  {trained_avg:+.2f}')
print(f'  Silent baseline:     {silent_avg:+.2f}')
print(f'  Beats baseline:      {trained_avg > silent_avg}')
print(f'  % ended safe:        {trained_safe_pct:.1f}%')

os.makedirs('results', exist_ok=True)
with open('results/trained_rewards.json', 'w') as f:
    json.dump({
        'policy': 'grpo_trained', 'n_episodes': 100,
        'avg_reward': round(trained_avg, 3),
        'pct_ended_safe': round(trained_safe_pct, 1),
        'episode_rewards': eval_results,
        'trust_trajectory': eval_trust,
        'outcomes': eval_outcomes,
        'rubric_history': rubric_history,
    }, f, indent=2)
print('Saved: results/trained_rewards.json')

## Cell 5 — Generate & Display Plots

Produces the four required plots and displays them inline.

In [ ]:
import json, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.makedirs('results/plots', exist_ok=True)

with open('results/baseline_rewards.json') as f: baseline = json.load(f)
with open('results/silent_rewards.json')  as f: silent   = json.load(f)
with open('results/trained_rewards.json') as f: trained  = json.load(f)

def smooth(data, w=20):
    return list(np.convolve(data, np.ones(w)/w, mode='valid')) if len(data) >= w else data

# ── Plot 1: Reward curve ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
b_rewards = [ep['total_reward'] for ep in baseline['episodes']]
t_rewards = trained['episode_rewards']
ax.plot(smooth(b_rewards), color='#e74c3c', linewidth=2,
        label=f"Random Agent (avg: {baseline['avg_reward']:+.1f})")
ax.fill_between(range(len(smooth(b_rewards))), smooth(b_rewards), alpha=0.08, color='#e74c3c')
if len(t_rewards) >= 20:
    ax.plot(smooth(t_rewards), color='#2ecc71', linewidth=2,
            label=f"GRPO Trained (avg: {trained['avg_reward']:+.1f})")
    ax.fill_between(range(len(smooth(t_rewards))), smooth(t_rewards), alpha=0.08, color='#2ecc71')
ax.axhline(silent['avg_reward'], color='#f39c12', linewidth=1.5, linestyle='--',
           label=f"Always-Silent ({silent['avg_reward']:+.2f})")
ax.set_xlabel('Episode', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Episode Reward', fontsize=13, fontweight='bold')
ax.set_title('SafeSignal — GRPO Training vs Random Baseline', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('results/plots/01_reward_curve.png', dpi=150, bbox_inches='tight')
plt.close()

# ── Plot 2: Trust comparison ───────────────────────────────────────────────
b_trust = [ep['final_guardian_trust'] for ep in baseline['episodes']]
t_trust = trained.get('trust_trajectory', [])
fig, ax = plt.subplots(figsize=(12, 5))
if len(b_trust) >= 20:
    ax.plot(smooth(b_trust), color='#e74c3c', linewidth=2,
            label=f"Random (avg: {np.mean(b_trust):.2f})")
if len(t_trust) >= 20:
    ax.plot(smooth(t_trust), color='#2ecc71', linewidth=2,
            label=f"Trained (avg: {np.mean(t_trust):.2f})")
ax.set_ylim(0, 1.1)
ax.set_xlabel('Episode', fontsize=13, fontweight='bold')
ax.set_ylabel('Final Guardian Trust', fontsize=13, fontweight='bold')
ax.set_title('SafeSignal — Guardian Trust Preservation', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('results/plots/02_trust_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

# ── Plot 3: Safety outcomes ────────────────────────────────────────────────
states = ['SAFE', 'VULNERABLE', 'AT_RISK', 'IN_DANGER']
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
b_counts = {s: 0 for s in states}
for ep in baseline['episodes']: b_counts[ep['final_hidden_state']] += 1
t_counts = {s: 0 for s in states}
for o in trained.get('outcomes', []): t_counts[o] += 1
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, counts, n, title in [
    (ax1, b_counts, len(baseline['episodes']), 'Random Agent'),
    (ax2, t_counts, len(trained.get('outcomes', [1])), 'GRPO Trained'),
]:
    pcts = [counts[s]/max(n,1)*100 for s in states]
    bars = ax.bar(states, pcts, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title}\nFinal Episode Outcomes', fontsize=13, fontweight='bold')
    ax.set_ylabel('% of Episodes', fontsize=12)
    ax.set_ylim(0, 105)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    for bar, pct in zip(bars, pcts):
        if pct > 2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                            f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')
fig.suptitle('SafeSignal — Child Safety Outcomes', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/03_safety_outcomes.png', dpi=150, bbox_inches='tight')
plt.close()

# ── Plot 4: Rubric breakdown ───────────────────────────────────────────────
rh = trained.get('rubric_history', [])
if rh:
    names  = ['intervention_timing', 'guardian_trust', 'silence_intelligence']
    labels = ['Intervention Timing (40%)', 'Guardian Trust (30%)', 'Silence Intelligence (20%)']
    cols   = ['#3498db', '#2ecc71', '#9b59b6']
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, name, label, col in zip(axes, names, labels, cols):
        scores   = [h.get(name, 0) for h in rh]
        smoothed = smooth(scores, w=min(20, len(scores)))
        ax.plot(smoothed, color=col, linewidth=2)
        ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.set_xlabel('Episode', fontsize=11)
        ax.set_ylabel('Rubric Score', fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.suptitle('Composable Rubric Scores During GRPO Training', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/plots/04_rubric_breakdown.png', dpi=150, bbox_inches='tight')
    plt.close()

print('All plots saved to results/plots/')

# Display inline
for fname in ['01_reward_curve.png', '02_trust_comparison.png',
              '03_safety_outcomes.png', '04_rubric_breakdown.png']:
    path = f'results/plots/{fname}'
    if os.path.exists(path):
        print(f'\n--- {fname} ---')
        display(Image(path))

## Results Summary

In [ ]:
print('=' * 55)
print('SAFESIGNAL GRPO TRAINING — RESULTS')
print('=' * 55)
print(f'  Random baseline:   {baseline["avg_reward"]:+.2f}')
print(f'  Silent baseline:   {silent["avg_reward"]:+.2f}  ← must beat this')
print(f'  Trained policy:    {trained["avg_reward"]:+.2f}')
print(f'  vs random:         {trained["avg_reward"] - baseline["avg_reward"]:+.2f}')
print(f'  vs silent:         {trained["avg_reward"] - silent["avg_reward"]:+.2f}')
print(f'  % ended safe:      {trained["pct_ended_safe"]}%')
print('=' * 55)
print(f'  Beat benchmark:    {trained["avg_reward"] > silent["avg_reward"]}')